## monitoring, drift, and the retraining trigger

three layers, detection separated from action:

| layer | signal | labels | window | acts as |
|---|---|---|---|---|
| 1 | data quality + ranges | no | same day | reject input, **never** retrain |
| 2 | persistence index | yes (t+1) | 60d | **the only** retrain trigger |
| 3 | evidently drift | no | 90d | evidence, not a pager |

why layer 3 is not a pager: at daily cadence a 30-day psi window fires above 0.2 about 80% of the time with no drift present (measured below). hourly data would fix that.

three scenarios, matching the project brief: baseline validation, drift response, and a corruption stress test.

In [ ]:
import sys, warnings
sys.path.insert(0, ".."); warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb, yaml

from streamflow import monitor
from streamflow.config import CONFIG, resolve
from streamflow.features import _feature_columns, TARGET_COLUMN as T

sienna, olive, steel, gray = "#a0522d", "#808000", "#4682b4", "#b0b0b0"
plt.rcParams["axes.facecolor"] = "#faf7f2"; plt.rcParams["figure.facecolor"] = "#faf7f2"

f = monitor.load_features()
cols = _feature_columns(f)
params = yaml.safe_load(open(resolve("config/best_params.yaml")))["params"]

def fit(train):
    return xgb.XGBRegressor(**params, objective="reg:squarederror",
                            random_state=42, n_jobs=-1).fit(train[cols], train[T])

print(f"{len(f):,} feature rows | {f.date.min().date()} -> {f.date.max().date()}")

### why psi is not our alarm
null test: two samples drawn from the *same* distribution, so no drift exists. a 30-day window still crosses psi 0.2 most of the time - that is the window, not the data.

In [ ]:
def psi(ref, cur, bins=10):
    qs = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(qs) < 3: return 0.0
    edges = np.concatenate(([-np.inf], qs[1:-1], [np.inf]))
    a = np.histogram(ref, bins=edges)[0] / len(ref)
    b = np.histogram(cur, bins=edges)[0] / len(cur)
    a, b = np.clip(a, 1e-4, None), np.clip(b, 1e-4, None)
    return float(((a - b) * np.log(a / b)).sum())

rng = np.random.default_rng(0)
hist = f[f.date < "2020-01-01"]["streamflow_t"].dropna().values
rows = []
for n in (30, 60, 90, 180, 365):
    v = np.array([psi(rng.choice(hist, 2000), rng.choice(hist, n)) for _ in range(200)])
    rows.append({"window_days": n, "median_psi": round(float(np.median(v)), 3),
                 "false_alarm_rate_at_0.2": f"{100*(v>0.2).mean():.0f}%"})
pd.DataFrame(rows).set_index("window_days")

### calibrating the retrain threshold
the threshold is derived, not borrowed. train once on pre-2016 data, score every day after, then take **non-overlapping** blocks (overlapping windows are autocorrelated and overstate the sample).

In [ ]:
backtest_model = fit(f[f.date < "2016-01-01"])
healthy = monitor.score_range("2016-01-01", model=backtest_model, features=f)

cal = pd.DataFrame([monitor.calibrate_pi_threshold(healthy, w) for w in (14, 30, 60, 90)])
cal[["window_days", "n_blocks", "median", "min", "share_below_zero"]].round(3).set_index("window_days")

a healthy model never put a 60-day block below zero in a decade, so **pi < 0 at 60 days** is both meaningful (worse than doing nothing) and safe. shorter windows are too noisy to alarm on.

### scenario 1 - baseline validation
clean data, current model. the monitor should stay quiet.

In [ ]:
current_model = fit(f[f.date < "2026-01-01"])
s_ok = monitor.score_range("2026-01-01", model=current_model, features=f)
pi_ok = monitor.rolling_pi(s_ok).dropna()

d_ok = monitor.decide(pi_ok.iloc[-1], {"passed": True})
print(f"rolling PI (60d) = {pi_ok.iloc[-1]:.3f}  ->  {d_ok['status']} / {d_ok['action']}")

### scenario 2 - drift response
a deliberately stale model (trained on january only) served through the rest of the year. this is an induced failure - the point is whether the monitor catches it and what it decides.

In [ ]:
stale_model = fit(f[(f.date >= "2026-01-01") & (f.date < "2026-02-01")])
s_stale = monitor.score_range("2026-02-01", model=stale_model, features=f)
pi_stale = monitor.rolling_pi(s_stale)

crossed = pi_stale[pi_stale < 0]
first = s_stale.loc[crossed.index[0], "date"] if len(crossed) else None
print(f"stale model PI at end: {pi_stale.dropna().iloc[-1]:.3f}")
print(f"first day PI crossed 0: {first.date() if first is not None else 'never'}")

d_stale = monitor.decide(pi_stale.dropna().iloc[-1], {"passed": True})
print(f"decision -> {d_stale['status']} / {d_stale['action']}")
print(f"  {d_stale['reason']}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(s_ok.date, monitor.rolling_pi(s_ok), color=steel, lw=1.4, label="current model")
ax.plot(s_stale.date, pi_stale, color=sienna, lw=1.4, label="stale (january-trained)")
ax.axhline(0, color=gray, ls="--", lw=1)
ax.text(s_stale.date.iloc[5], 0.02, "PI = 0: no better than persistence", fontsize=8, color="#555")
ax.set_ylabel("rolling PI (60d)"); ax.set_title("the retrain trigger", weight="bold")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

**retrain and re-check.** the fix for staleness is fresher data with the same hyperparameters - not a new search.

In [ ]:
cut = pd.Timestamp("2026-06-01")                       # retrain on everything up to the alert
retrained = fit(f[f.date < cut])
holdout = f[f.date >= cut]

rows = []
for name, m in [("stale (champion)", stale_model), ("retrained (candidate)", retrained)]:
    s = monitor.score_range(cut, model=m, features=f)
    rows.append({"model": name, "PI on holdout": round(monitor.persistence_index(s), 3)})
gate = pd.DataFrame(rows).set_index("model")
gate["promote?"] = ["", "yes" if gate['PI on holdout'].iloc[1] > gate['PI on holdout'].iloc[0] else "no"]
gate

the candidate only ships if it beats the champion on held-out days it never saw. a retrain triggered for a bad reason cannot silently degrade production.

### scenario 3 - corruption stress test
the brief asks for artificially corrupted inputs. two defence layers: the api contract rejects out-of-range and schema violations outright (see `03_serving_prototype`), and evidently catches shifts that are *valid* but wrong.

first calibrate: run the report over known-clean historical windows and confirm it stays quiet. psi 0.2 is the textbook cutoff but **every** clean 90-day window here exceeds it, so per-column thresholds are set above the clean maximum (`config.monitoring.psi_thresholds`).

In [ ]:
s_all = monitor.score_range("2015-01-01", model=current_model, features=f)
m = monitor.with_predictions(f, s_all)
cur = m[m.date > m.date.max() - pd.Timedelta(days=90)]
ref = monitor.seasonal_reference(m, cur)              # same season, earlier years

# calibration: how often does a known-clean window alarm?
alarms = tot = 0
for end in pd.date_range("2019-06-01", "2026-06-01", freq="6MS"):
    c = m[(m.date > end - pd.Timedelta(days=90)) & (m.date <= end)]
    r_ = monitor.seasonal_reference(m, c)
    if len(c) < 80 or len(r_) < 200: continue
    tot += 1; alarms += monitor.drift_report(r_, c)["dataset_drift"]
print(f"clean windows alarming: {alarms}/{tot}  ({100*alarms/max(tot,1):.0f}% false-alarm rate)")

scenarios = {"clean": cur}
shift = cur.copy(); shift["tmax_c"] -= 15                       # january temps in summer
scenarios["temp shifted -15C"] = shift
wet = cur.copy(); wet["precip_mm"] = wet["precip_mm"] * 5 + 20  # inflated rainfall
scenarios["precip inflated"] = wet

out = []
for name, frame in scenarios.items():
    r = monitor.drift_report(ref, frame, f"../reports/stress_{name.split()[0]}.html")
    out.append({"scenario": name, "flagged": r["dataset_drift"],
                **{k: round(v, 2) for k, v in r["per_column_psi"].items()}})
pd.DataFrame(out).set_index("scenario")

the reference matters as much as the test: against a plain calendar-year baseline every summer window looks 100% drifted purely because of the season. matching on time of year removes the calendar and leaves real signal.

### the system pillar
session 7: *"most failures are not model failures, they are system failures."* data and model signals alone miss that whole class, so every run also records model age, numeric stability and compute cost - ml test score monitors 4, 5 and 6.

these are logged even when healthy: a slow leak in latency or a creeping model age is only visible as a trend.

(`model_age_days` is NaN here because these cells fit models locally; in production it comes from the mlflow registry, as in `python -m streamflow.monitor`.)

In [ ]:
import time
t0 = time.perf_counter()
s_sys = monitor.score_range("2026-01-01", model=current_model, features=f)
health = monitor.system_health(s_sys, time.perf_counter() - t0)
pd.Series(health).to_frame("value")

the same signals decide differently - a model emitting NaN is broken, not stale, so it is investigated rather than retrained:

In [ ]:
cases = {
    "healthy":            dict(pi=0.35, quality={"passed": True}, system={"model_age_days": 3, "nonfinite_predictions": 0}),
    "model too old":      dict(pi=0.35, quality={"passed": True}, system={"model_age_days": 400, "nonfinite_predictions": 0}),
    "NaN predictions":    dict(pi=0.35, quality={"passed": True}, system={"nonfinite_predictions": 12}),
    "broken feed":        dict(pi=-9.0, quality={"passed": False, "reason": "upstream stale"}, system={"model_age_days": 400}),
}
pd.DataFrame([{"scenario": k, **{f_: monitor.decide(v["pi"], v["quality"], system=v["system"])[f_]
                                 for f_ in ("status", "action")}} for k, v in cases.items()]).set_index("scenario")

### the alert log
every decision is recorded, including the quiet ones - a log of only failures cannot show a stable baseline.

In [ ]:
for label, d in [("baseline", d_ok), ("stale model", d_stale),
                 ("flood-like drift", monitor.decide(0.35, {"passed": True},
                                                     drift={"dataset_drift": True, "drift_share": 0.8})),
                 ("bad feed", monitor.decide(-2.0, {"passed": False, "reason": "stale upstream data"}))]:
    monitor.log_alert(d, at=pd.Timestamp("2026-08-01"))

log = pd.read_parquet(resolve(CONFIG["monitoring"]["alerts_path"]))
log[["as_of", "status", "action", "pi", "reason"]].tail(4)

### takeaways
- **pi < 0 on a 60-day window** is the retrain trigger: calibrated, not borrowed - a healthy model never crossed it in a 10.5-year backtest
- **drift alone never retrains.** a flood shifts the inputs while the model is fine; retraining on it would teach the rarest data as normal
- **data-quality failures never retrain either** - retraining cannot fix a broken feed
- **the reference must match the season**, or every summer reads as drift
- at daily cadence drift detection is evidence; at hourly resolution a 90-day window holds 24x the rows and could become a real alarm